# An exact test and its power

A support chatbot resolves 70% of customer tickets without a human. A new version is
tested on $n$ fresh tickets, and it ships only if it resolves enough of them to beat
70% in a one-sided test at the 5% level. The normal approximation from the lesson is
close but not exact for a few dozen tickets. Here you compute the test exactly from the
binomial distribution, then ask the planning question: if the new version really
resolves 80%, how many tickets make it likely to pass?

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
from math import comb, sqrt, erf

def normal_cdf(z):
    return 0.5 * (1 + erf(z / sqrt(2)))

P0 = 0.70      # the old version's resolution rate, the null hypothesis
P1 = 0.80      # the rate the new version is hoped to have
ALPHA = 0.05   # the test's level

## Part 1 (coding task): The exact tail

Write `tail_prob(k, n, p)`, the chance that a binomial count with $n$ trials and success
probability $p$ is **at least** $k$:

$$
P(X \ge k) = \sum_{j=k}^{n} \binom{n}{j} p^j (1 - p)^{n-j} .
$$

It must work for any integer $k$: for $k \le 0$ the answer is 1, and for $k > n$ it is 0.
`comb(n, j)` from `math` gives $\binom{n}{j}$. Keep $n$ at 1000 or less.

In [ ]:
def tail_prob(k, n, p):
    # TODO
    raise NotImplementedError

In [ ]:
assert abs(tail_prob(2, 3, 0.5) - 0.5) < 1e-12, (
    f"tail_prob(2, 3, 0.5) should be P(X >= 2) = (3 + 1) / 8 = 0.5, got {tail_prob(2, 3, 0.5)}"
    + (": that is P(X > 2), include the count k itself" if abs(tail_prob(2, 3, 0.5) - 0.125) < 1e-12 else ""))
assert abs(tail_prob(2, 3, 0.9) - 0.972) < 1e-12, (
    f"tail_prob(2, 3, 0.9) should be 3(0.81)(0.1) + 0.729 = 0.972, got {tail_prob(2, 3, 0.9)}"
    + (": p and 1 - p are swapped" if abs(tail_prob(2, 3, 0.9) - 0.028) < 1e-12 else ""))
assert abs(tail_prob(0, 3, 0.9) - 1) < 1e-12 and abs(tail_prob(-2, 3, 0.9) - 1) < 1e-12, "every count is at least 0, so tail_prob(k, n, p) is 1 for k <= 0"
assert abs(tail_prob(4, 3, 0.9)) < 1e-12, "a count of 3 trials cannot reach 4, so tail_prob(4, 3, p) is 0"
assert abs(tail_prob(17, 20, 0.6) - 0.015961) < 1e-6, f"tail_prob(17, 20, 0.6) should be about 0.015961, got {tail_prob(17, 20, 0.6)}"
print(f"P(X >= 49) for n = 60 tickets at 70%: {tail_prob(49, 60, P0):.4f}")
print("Part 1 passed")

## Part 2 (coding task): The passing score

Write `critical_score(n, p0, alpha)`: the smallest number of resolved tickets $k$ for
which a version with rate $p_0$ would reach $k$ or more with probability at most
`alpha`. A version passes the test when it resolves at least that many. If no $k$ from 0
to $n$ qualifies, return $n + 1$ (no score passes).

In [ ]:
def critical_score(n, p0, alpha=0.05):
    # TODO
    raise NotImplementedError

In [ ]:
k60 = critical_score(60, P0, ALPHA)
k100 = critical_score(100, 0.75, 0.05)
if k60 == 48:
    if k100 == 82:
        why = ": the tail compared with alpha must be P(X >= k), which includes k itself, not P(X > k)"
    elif k100 == 83:
        why = ": the normal approximation's cut, 47.84, rounds up to 48, so use the exact tail instead"
    else:
        why = ""
    raise AssertionError(f"critical_score(60, 0.7) gave 48, but P(X >= 48) = {tail_prob(48, 60, P0):.4f} is above 0.05{why}")
assert k60 != 50, "critical_score(60, 0.7) gave 50, one too many: 49 already has a tail of at most 0.05"
assert k60 == 49, f"critical_score(60, 0.7) should be 49, got {k60}"
assert k100 == 83, f"critical_score(100, 0.75) should be 83, got {k100}"
assert critical_score(3, 0.5, 0.05) == 4, "with 3 fair coin flips even 3 of 3 has chance 0.125 > 0.05, so no score passes: return n + 1 = 4"
assert critical_score(60, P0, 0.01) == 51, f"critical_score(60, 0.7, alpha=0.01) should be 51, got {critical_score(60, P0, 0.01)}: use the alpha argument"
print(f"n = 60: pass with {k60} or more resolved")
print("Part 2 passed")

## Part 3 (coding task): The chance of passing

The **power** of the test against a rate $p_1$ is the probability that the test, run on
$n$ tickets at level `alpha` against $p_0$, passes a version whose true resolution rate is
$p_1$. Write `power(n, p0, p1, alpha)`.

In [ ]:
def power(n, p0, p1, alpha=0.05):
    # TODO
    raise NotImplementedError

In [ ]:
pw = power(60, P0, P1, ALPHA)
assert abs(pw - tail_prob(49, 60, 0.7)) > 1e-6, "this is the chance under p0, at most alpha by design: compute the tail at the true rate p1"
assert abs(pw - tail_prob(critical_score(60, 0.8), 60, 0.8)) > 1e-6, "the passing score must come from p0, the rate the test is checked against, not from p1"
assert abs(pw - 0.448617) < 1e-5, f"power(60, 0.7, 0.8) should be about 0.4486, got {pw:.6f}"
assert abs(power(100, 0.75, 0.85) - 0.763277) < 1e-5, f"power(100, 0.75, 0.85) should be about 0.7633, got {power(100, 0.75, 0.85):.6f}"
print(f"with 60 tickets, an 80% version passes {pw:.1%} of the time")
print("Part 3 passed")

## Part 4 (coding task): How many tickets to test

You want a test size that keeps its power: write `smallest_n(p0, p1, target, alpha, n_max)`,
the smallest $n$ such that `power(m, p0, p1, alpha)` is at least `target` for every $m$
from $n$ to `n_max`. Return `None` if the power at `n_max` itself is below `target`.

In [ ]:
def smallest_n(p0, p1, target=0.8, alpha=0.05, n_max=300):
    # TODO
    raise NotImplementedError

In [ ]:
n80 = smallest_n(P0, P1, 0.8, ALPHA, n_max=200)
assert n80 is not None, "smallest_n(0.7, 0.8, n_max=200) returned None, but the power at n = 200 is above 0.8"
assert n80 != 119, (f"119 is the first n whose power reaches 0.8, but power(120) = {power(120, P0, P1):.3f} is below it again: "
                    "return the n from which the power stays at or above the target all the way to n_max")
assert n80 == 130, f"smallest_n(0.7, 0.8, n_max=200) should be 130, got {n80}"
assert smallest_n(P0, P1, 0.8, ALPHA, n_max=128) == 127, f"with n_max = 128 the power holds from 127 on, got {smallest_n(P0, P1, 0.8, ALPHA, n_max=128)}"
assert smallest_n(P0, P1, 0.8, ALPHA, n_max=125) is None, f"power(125) = {power(125, P0, P1):.3f} is below 0.8, so with n_max = 125 return None"
n95 = smallest_n(0.9, 0.95, 0.8, ALPHA, n_max=250)
assert n95 != 179, "179 is the first n whose power reaches 0.8 for 0.9 against 0.95: it does not stay there"
assert n95 == 203, f"smallest_n(0.9, 0.95, n_max=250) should be 203, got {n95}"
print("power for n = 115 to 130:")
print("  " + "  ".join(f"{n}:{power(n, P0, P1):.3f}" for n in range(115, 131)))
print("Part 4 passed")

## Solution

### Part 1: The exact tail

In [ ]:
def tail_prob(k, n, p):
    k = max(int(k), 0)                         # P(X >= k) is 1 for k <= 0
    return sum(comb(n, j) * p**j * (1 - p)**(n - j) for j in range(k, n + 1))

**Why it works.** The event "at least $k$" is the union of the separate counts $k, k + 1, \dots, n$, which cannot happen together, so their probabilities add. Clipping $k$ at 0 makes every $k \le 0$ give the full sum, 1, and for $k > n$ the range is empty and the sum is 0. Starting the sum at $k + 1$ instead would give $P(X > k)$: for 17 or more successes in 20 trials at 60% that is 0.0036 rather than the true 0.0160, more than four times too small.

### Part 2: The passing score

In [ ]:
def critical_score(n, p0, alpha=0.05):
    k, tail = n + 1, 0.0                       # P(X >= n + 1) = 0
    while k > 0:
        p_next = comb(n, k - 1) * p0**(k - 1) * (1 - p0)**(n - k + 1)
        if tail + p_next > alpha:              # one more count would push the tail over alpha
            break
        tail += p_next
        k -= 1
    return k

**Why it works.** The tail $P(X \ge k)$ only grows as $k$ falls, so the code starts above $n$, where the tail is 0, and adds one count at a time from the top. It stops just before the tail would pass `alpha`, which leaves the smallest $k$ whose tail is at most `alpha`. Scanning up from $k = 0$ with `tail_prob` gives the same answer with more arithmetic. With 60 tickets the normal approximation's cut is 47.84, which rounds up to 48, but the exact chance of 48 or more at 70% is 0.057, over the 5% the test allows. The exact test needs 49. Near the cut the binomial's right tail is a little heavier than the normal curve's.

### Part 3: The chance of passing

In [ ]:
def power(n, p0, p1, alpha=0.05):
    k = critical_score(n, p0, alpha)           # the cut is set under the null hypothesis
    return tail_prob(k, n, p1)                 # the chance of clearing it is computed under p1

**Why it works.** The test fixes its passing score using $p_0$ alone, so a version that is no better passes at most 5% of the time. Whether a better version passes is then a question about its own count, a binomial with rate $p_1$, reaching that score. With 60 tickets a truly 80% version passes only 45% of the time: a real 10-point gain is more likely missed than found.

### Part 4: How many tickets to test

In [ ]:
def smallest_n(p0, p1, target=0.8, alpha=0.05, n_max=300):
    if power(n_max, p0, p1, alpha) < target:
        return None
    n = n_max
    while n > 1 and power(n - 1, p0, p1, alpha) >= target:   # walk down while the power holds
        n -= 1
    return n

**Why it works.** The power curve is a sawtooth. The passing score is a whole number, so as $n$ grows by one the score sometimes stays put and sometimes jumps up by one, and at each jump the power drops a little. For 70% against 80% it first reaches 0.804 at $n = 119$, falls to 0.790 at 120 and dips below 0.8 several more times, the last at 129, and holds from 130 on. Walking down from `n_max` while the power stays at or above the target finds that point directly. Stopping at the first $n$ that reaches the target would plan a test of 119 tickets, and a test of 120 would then be less likely to pass than the plan says.